In [0]:

-- update the dummy records 
merge into data.late_dim.dim_customer t
using data.late_dim.src_customer s
on s.customer_id = t.customer_id
when matched and t.customer_name= 'unknown' and t.contact is null and t.email is null 
then update set 
  t.customer_name = s.customer_name,
  t.customer_city = s.customer_city,
  t.email = s.email,
  t.contact = s.contact,
  t.start_date = s.created_date,
  t.is_active = True,
  t.ingestion_time = current_timestamp();

-- update old records
merge into data.late_dim.dim_customer t
using data.late_dim.src_customer s
on s.customer_id = t.customer_id and t.is_active = True
when matched 
and (s.contact != t.contact 
      or s.email != t.email 
      or s.customer_city != t.customer_city
      or s.customer_name != t.customer_name)
then update set
  t.is_active = False,
  t.end_date = s.updated_date;


INSERT INTO data.late_dim.dim_customer (
  customer_id, customer_name, customer_city, email, contact,
  start_date, end_date, is_active, ingestion_time
)
SELECT 
  s.customer_id, s.customer_name, s.customer_city, s.email, s.contact,
  s.created_date, s.updated_date, true, current_timestamp()
FROM data.late_dim.src_customer s
LEFT JOIN data.late_dim.dim_customer t
  ON s.customer_id = t.customer_id AND t.is_active = true
WHERE 
  t.customer_id IS NULL
  OR (
    s.contact <> t.contact OR
    s.email <> t.email OR
    s.customer_city <> t.customer_city OR
    s.customer_name <> t.customer_name
  );